# Procesos Gaussianos — Clasificador (GPC)

**Repositorio:** `autodidact/both/gaussian_processes/`

---

## 1. Idea General e Intuición

La **Clasificación con Procesos Gaussianos (GPC)** extiende el GPR al problema de clasificación. El desafío: las probabilidades de clase deben estar en $[0,1]$, pero un GP produce valores en $\mathbb{R}$.

**Solución:** Colocar un GP como función latente $f(\mathbf{x})$ y pasar la salida por una función de enlace (sigmoide para binario, softmax para multiclase):

$$p(y=1 \mid \mathbf{x}) = \sigma(f(\mathbf{x}))$$

**Complicación:** A diferencia de la regresión (donde el posterior es gaussiano exacto), en clasificación el posterior **no es gaussiano** (la likelihood binomial/multinomial no es conjugada con el prior GP). Se requieren métodos de aproximación.

## 2. Fundamento Matemático

### 2.1 Modelo Generativo

$$f \sim \mathcal{GP}(0, k(\mathbf{x}, \mathbf{x}'))$$
$$p(y \mid f) = \text{Bernoulli}(\sigma(f))$$

La distribución posterior sobre $f$ dados los datos:

$$p(f \mid \mathbf{X}, \mathbf{y}) = \frac{p(\mathbf{y} \mid f) \cdot p(f \mid \mathbf{X})}{p(\mathbf{y} \mid \mathbf{X})}$$

Esta no tiene forma cerrada gaussiana → se aproxima.

### 2.2 Aproximación de Laplace (usada en scikit-learn)

scikit-learn usa la **aproximación de Laplace**: aproxima el posterior por una gaussiana centrada en el MAP estimate $\hat{f}$:

1. Encontrar $\hat{f} = \text{argmax}_f \log p(\mathbf{y} \mid f) + \log p(f)$ (Newton-Raphson).
2. Aproximar el posterior: $p(f \mid \mathbf{X}, \mathbf{y}) \approx \mathcal{N}(\hat{f}, \mathbf{A}^{-1})$.
   Donde $\mathbf{A} = -\nabla^2 \log p(\mathbf{y} \mid \hat{f}) + \mathbf{K}^{-1}$.
3. Predecir marginalizando sobre la función latente:
$$p(y_* = 1 \mid \mathbf{x}_*, \mathbf{X}, \mathbf{y}) = \int \sigma(f_*) \cdot p(f_* \mid \mathbf{x}_*, \mathbf{X}, \mathbf{y}) \, df_*$$

### 2.3 Predicción — Probabilidades de Clase

La probabilidad predictiva se obtiene por integración numérica:

$$\hat{p}_* = \mathbb{E}_{f_* \mid \cdot}[\sigma(f_*)] = \int \sigma(f_*) \mathcal{N}(f_* \mid \mu_*, \sigma_*^2) \, df_*$$

Donde $\mu_*$ y $\sigma_*^2$ son la media y varianza del posterior en $\mathbf{x}_*$.

### 2.4 Complejidad

| Operación | Complejidad |
|---|---|
| Entrenamiento | $O(n^3)$ por iteración Newton |
| Predicción | $O(n^2)$ por punto |

> **Nota:** GPC es más lento que GPR por las iteraciones de Newton-Raphson.

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Dataset **pequeño** y necesitas **probabilidades de clase bien calibradas**.
- Quieres **incertidumbre en la clasificación** (no solo la clase, sino qué tan seguro está).
- Las regiones de decisión son **no lineales y complejas**.
- En **Optimización Bayesiana** para problemas de clasificación.

### ❌ NO lo uses cuando:
- Dataset grande (O(n³) es prohibitivo).
- Las clases son **claramente separables linealmente** (usar LogReg).
- Necesitas **predicciones en tiempo real** rápidas.
- Hay **muchas clases** (multiclase aproximado, más costoso).

## 4. Métricas de Evaluación

| Métrica | Por qué es relevante para GPC |
|---|---|
| **ROC-AUC** | Evalúa el ranking de probabilidades calibradas |
| **Log-loss** | Mide calibración de las probabilidades — clave para GPC |
| **Brier Score** | $\frac{1}{n}\sum(y_i - \hat{p}_i)^2$ — combina calibración y discriminación |
| **Accuracy** | Métrica de clasificación estándar |

**Brier Score:**
$$BS = \frac{1}{n}\sum_{i=1}^{n}(\hat{p}_i - y_i)^2 \in [0, 1]$$

Menor es mejor; 0 = perfecto.

## 5. Problemas y Casos Raros

### 5.1 Probabilidades Extremas (0 o 1)
En regiones alejadas de los datos de entrenamiento, $\sigma_*^2$ es grande pero $\hat{p}_*$ puede ser 0.5 (máxima incertidumbre). Esto es correcto matemáticamente pero puede sorprender.

### 5.2 Convergencia Newton-Raphson
El algoritmo de Laplace puede no converger con ciertos kernels o configuraciones.
**Solución:** Probar kernels más suaves (RBF vs Matérn 1.5).

### 5.3 Multiclase
scikit-learn implementa GPC multiclase como OvR (One-vs-Rest) con aproximación de Laplace independiente por clase — esto no es el posterior conjunto correcto.

### 5.4 Calibración vs GPR
GPC ya produce probabilidades "bayesianas", pero la aproximación de Laplace puede desplazarlas. Verificar con la curva de calibración.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, WhiteKernel, ConstantKernel as C
)
from sklearn.datasets import load_breast_cancer, load_iris, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay,
    log_loss, brier_score_loss
)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
rng  = np.random.RandomState(SEED)
print('✅ Librerías cargadas.')

---
## 6. Demostración 2D — Probabilidad de Clase e Incertidumbre

In [ ]:
# =============================================================================
# DATOS MAKE_MOONS: visualizar probabilidad de clase e INCERTIDUMBRE del GPC
# La incertidumbre es máxima en la frontera de decisión y lejos de los datos
# =============================================================================
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=200, noise=0.25, random_state=SEED)
scaler_m = StandardScaler()
X_moons_sc = scaler_m.fit_transform(X_moons)

kernel_gpc = C(1.0) * RBF(1.0)
gpc_vis = GaussianProcessClassifier(kernel=kernel_gpc,
                                     n_restarts_optimizer=5,
                                     random_state=SEED)
gpc_vis.fit(X_moons_sc, y_moons)

h = 0.05
x0_min, x0_max = X_moons_sc[:,0].min()-1, X_moons_sc[:,0].max()+1
x1_min, x1_max = X_moons_sc[:,1].min()-1, X_moons_sc[:,1].max()+1
xx, yy = np.meshgrid(np.arange(x0_min, x0_max, h), np.arange(x1_min, x1_max, h))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

proba_grid = gpc_vis.predict_proba(grid_pts)[:, 1].reshape(xx.shape)
# Incertidumbre: entropía de la distribución predictiva
entropy_grid = -(proba_grid * np.log(proba_grid + 1e-10) +
                 (1-proba_grid) * np.log(1-proba_grid + 1e-10)).reshape(xx.shape)

cmap_pts = ListedColormap(['#CC0000', '#0000CC'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probabilidad de clase 1
cf = axes[0].contourf(xx, yy, proba_grid, levels=np.linspace(0, 1, 21), cmap='RdBu_r')
axes[0].scatter(X_moons_sc[:,0], X_moons_sc[:,1], c=y_moons, cmap=cmap_pts,
               edgecolors='k', s=40, zorder=5)
axes[0].contour(xx, yy, proba_grid, levels=[0.5], colors='black', linewidths=2)
plt.colorbar(cf, ax=axes[0], label='P(y=1|x)')
axes[0].set_title('Probabilidad Predictiva P(y=1|x)', fontsize=12, fontweight='bold')

# Incertidumbre (entropía)
cf2 = axes[1].contourf(xx, yy, entropy_grid, levels=20, cmap='YlOrRd')
axes[1].scatter(X_moons_sc[:,0], X_moons_sc[:,1], c=y_moons, cmap=cmap_pts,
               edgecolors='k', s=40, zorder=5)
plt.colorbar(cf2, ax=axes[1], label='Entropía H(p)')
axes[1].set_title('Incertidumbre Predictiva (Entropía)', fontsize=12, fontweight='bold')

plt.suptitle('GPC — Probabilidad e Incertidumbre (make_moons)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Clasificación Real — Breast Cancer

In [ ]:
# =============================================================================
# CARGA Y SPLIT
# =============================================================================
bc = load_breast_cancer(as_frame=True)
X, y = bc.data.values, bc.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
scaler_bc = StandardScaler()
X_tr_sc = scaler_bc.fit_transform(X_train)
X_te_sc = scaler_bc.transform(X_test)
print(f'Train: {X_tr_sc.shape} | Test: {X_te_sc.shape}')

In [ ]:
# =============================================================================
# MODELO GPC
# copy_X_train=False: no guardar copia de los datos de train (ahorra memoria)
# warm_start: reusar la solución anterior como punto de inicio
# =============================================================================
kernel_bc = C(1.0, (1e-2, 1e2)) * RBF(length_scale=np.ones(X_tr_sc.shape[1]),
                                        length_scale_bounds=(1e-2, 1e2))

gpc = GaussianProcessClassifier(
    kernel=kernel_bc,
    n_restarts_optimizer=3,
    max_iter_predict=200,
    random_state=SEED
)
gpc.fit(X_tr_sc, y_train)

y_pred  = gpc.predict(X_te_sc)
y_proba = gpc.predict_proba(X_te_sc)[:, 1]

print(f'Accuracy   : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC    : {roc_auc_score(y_test, y_proba):.4f}')
print(f'Log-loss   : {log_loss(y_test, y_proba):.4f}')
print(f'Brier Score: {brier_score_loss(y_test, y_proba):.4f}')
print(f'\nKernel optimizado: {gpc.kernel_}')
print()
print(classification_report(y_test, y_pred, target_names=bc.target_names))

In [ ]:
# =============================================================================
# VISUALIZACIÓN: ROC + Calibración + Matriz de Confusión
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ROC
RocCurveDisplay.from_estimator(gpc, X_te_sc, y_test, ax=axes[0], name='GPC')
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('Curva ROC', fontweight='bold')

# Calibración
CalibrationDisplay.from_estimator(gpc, X_te_sc, y_test,
                                   n_bins=8, ax=axes[1], name='GPC')
axes[1].set_title('Calibración de Probabilidades', fontweight='bold')

# Conf. Matrix
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=bc.target_names).plot(
    ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Matriz de Confusión', fontweight='bold')

plt.suptitle('GPC — Breast Cancer', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Clasificación Multiclase — Iris

In [ ]:
# =============================================================================
# GPC MULTICLASE (OvR): scikit-learn usa un GPC binario por clase
# =============================================================================
iris = load_iris(as_frame=True)
X_ir, y_ir = iris.data.values, iris.target

X_tr_ir, X_te_ir, y_tr_ir, y_te_ir = train_test_split(
    X_ir, y_ir, test_size=0.2, random_state=SEED, stratify=y_ir
)
scaler_ir = StandardScaler()
X_tr_ir_sc = scaler_ir.fit_transform(X_tr_ir)
X_te_ir_sc = scaler_ir.transform(X_te_ir)

gpc_multi = GaussianProcessClassifier(
    kernel=C(1.0) * RBF(1.0),
    n_restarts_optimizer=3,
    multi_class='one_vs_rest',
    random_state=SEED
)
gpc_multi.fit(X_tr_ir_sc, y_tr_ir)
y_pred_ir = gpc_multi.predict(X_te_ir_sc)

print(f'Accuracy multiclase: {accuracy_score(y_te_ir, y_pred_ir):.4f}')
print(classification_report(y_te_ir, y_pred_ir, target_names=iris.target_names))

# Frontera de decisión (PCA 2D)
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=SEED)
X_tr_2d = pca.fit_transform(X_tr_ir_sc)
X_te_2d = pca.transform(X_te_ir_sc)

gpc_2d = GaussianProcessClassifier(kernel=C(1.0)*RBF(1.0), random_state=SEED)
gpc_2d.fit(X_tr_2d, y_tr_ir)

h = 0.05
xx, yy = np.meshgrid(np.arange(X_tr_2d[:,0].min()-1, X_tr_2d[:,0].max()+1, h),
                     np.arange(X_tr_2d[:,1].min()-1, X_tr_2d[:,1].max()+1, h))
Z = gpc_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

cmap_bg  = ListedColormap(['#FFAAAA','#AAFFAA','#AAAAFF'])
cmap_pts = ListedColormap(['#CC0000','#00AA00','#0000CC'])

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_bg)
ax.scatter(X_tr_2d[:,0], X_tr_2d[:,1], c=y_tr_ir, cmap=cmap_pts,
           edgecolors='k', s=50, label='Train')
ax.scatter(X_te_2d[:,0], X_te_2d[:,1], c=y_te_ir, cmap=cmap_pts,
           edgecolors='white', s=80, marker='^', label='Test')
ax.set_title('GPC Multiclase (OvR) — Iris (PCA 2D)', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# COMPARATIVA: GPC vs Logistic Regression vs SVC (con probabilidades)
# Usando CV para comparación justa
# =============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

models_compare = {
    'GPC (RBF)':   GaussianProcessClassifier(kernel=C(1.0)*RBF(1.0), random_state=SEED),
    'LogReg (L2)': Pipeline([('sc', StandardScaler()),
                              ('lr', LogisticRegression(C=1.0, random_state=SEED))]),
    'SVC (RBF)':   Pipeline([('sc', StandardScaler()),
                              ('svc', SVC(kernel='rbf', probability=True, random_state=SEED))])
}

scaler_compare = StandardScaler()
X_sc = scaler_compare.fit_transform(X)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f'{"Modelo":<16}  {"ROC-AUC":>12}  {"Log-loss":>12}  {"Accuracy":>10}')
print('-' * 55)

# GPC necesita datos ya escalados
for name, model in models_compare.items():
    X_input = X_sc if name == 'GPC (RBF)' else X
    roc  = cross_val_score(model, X_input, y, cv=skf, scoring='roc_auc')
    lls  = -cross_val_score(model, X_input, y, cv=skf, scoring='neg_log_loss')
    acc  = cross_val_score(model, X_input, y, cv=skf, scoring='accuracy')
    print(f'{name:<16}  {roc.mean():.4f}±{roc.std():.3f}  '
          f'{lls.mean():.4f}±{lls.std():.3f}  {acc.mean():.4f}±{acc.std():.3f}')

---
## 9. Resumen Comparativo GPR vs GPC

| Aspecto | GPR (Regresión) | GPC (Clasificación) |
|---|---|---|
| **Posterior** | Gaussiano exacto | Aproximado (Laplace) |
| **Likelihood** | Gaussiana | Bernoulli / Multinomial |
| **Salida** | Media ± std | Probabilidades de clase |
| **Métrica clave** | RMSE + Cobertura IC | ROC-AUC + Brier Score |
| **Complejidad** | O(n³) | O(n³) × iteraciones Newton |

### ✅ Checklist GPC:
- [ ] ¿Escalé las features?
- [ ] ¿Verifiqué la calibración con `CalibrationDisplay`?
- [ ] ¿El dataset tiene ≤ 5000 muestras? (más lento que GPR)
- [ ] ¿Comparé LML de distintos kernels?
- [ ] ¿Calculé el Brier Score además del ROC-AUC?

---
*Ver notebook de regresión: `gp_regressor.ipynb`*